# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 付録：Rを単体で回すより、パイプラインに組んだほうが実務的か


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Landau, W. M. (2021). The targets R package: a dynamic Make-like function-oriented pipeline toolkit for reproducibility and high-performance computing. *Journal of Open Source Software*, 6(57), 2959。

■ 問い：ノートブックを毎回上から実行するのと、パイプラインを組むのはどちらが良いか：

本書の第0〜16章は、学習用に「1つのノートブックを上から実行する」形をとっている。初学者にとってはこれが最短距離で、章をまたぐ依存を気にせず読めるという利点がある。

しかし、実際の研究で同じ前処理（第1章）→複数の統計モデル（第3〜9章）→図表出力、という工程を**繰り返す**ようになったら話は別である。毎回全セルを再実行すると、

1. 変更していない前処理まで毎回再計算して時間を無駄にする
2. どのモデルがどのデータ・どのバージョンのコードから作られたか追跡できなくなる（再現性の喪失）
3. 複数の解析を並行して試すとき、ノートブックのセル順序に解析の依存関係を人間が手で管理することになる

という問題が出る。**答えは「単体実行→最後にパイプライン化」の順**である。本書のように1トピック1ノートブックで学ぶ段階では単体実行のままでよいが、本格的な研究に入ったら`targets`パッケージへの移行を検討するとよい。

この付録では、実際に`targets`をインストールし、第1章のデータ生成・第5章の線形混合モデル・第8章のベイズモデルを1本のパイプラインにまとめ、`tar_make()`を最後まで走らせて結果を確認する。テキストで示すだけのサンプルコードではなく、このノートブック単体で動く。

In [ ]:
# ⏳ 数十秒〜数分。
install.packages(c("targets", "tarchetypes", "visNetwork"), repos = "https://cloud.r-project.org")
library(targets)


■ まず、パイプラインが読み込む元データを作る：

`targets`はファイルの変更を検知して再計算するかどうかを決めるので、まず第1章と同じ手順で`vot_f0_synthetic.csv`をこの場に生成しておく。

In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、ランダム効果の分散推定値が0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。

write_csv(vot_data, "vot_f0_synthetic.csv")  # targetsが監視するファイル


■ targetsパッケージの考え方：

`targets`は「入力が変わっていない処理は再実行しない」ことを保証するRのパイプラインツールである（GNU Makeの考え方をRの関数に適用したもの）。プロジェクト直下に`_targets.R`という設定ファイルを1つ置き、処理をターゲット（関数呼び出し）の連鎖として宣言する。

In [ ]:
# プロジェクトのルートに保存する _targets.R を、このセルから書き出す。
pipeline_example <- '
library(targets)
library(tarchetypes)

tar_option_set(packages = c("tidyverse", "lme4", "brms"))

list(
  # 第1章: データ読み込み・前処理（このターゲットの出力はvot_f0_synthetic.csvの内容が変わらない限り再計算されない）
  tar_target(vot_data_file, "vot_f0_synthetic.csv", format = "file"),
  tar_target(vot_data, read_csv(vot_data_file, show_col_types = FALSE) %>%
    mutate(
      speaker = factor(speaker), word = factor(word),
      gender = factor(gender),
      condition = factor(condition, levels = c("voiced", "voiceless"))
    )
  ),

  # 第5章: 線形混合効果モデル（vot_dataが変わったときだけ再フィットされる）
  tar_target(model_lmm, lmer(
    VOT ~ condition + (1 + condition | speaker) + (1 | word),
    data = vot_data
  )),

  # 第8章: ベイズモデル（重い計算なので、変更がなければ絶対に再実行したくない）。
  # 事前分布は第8章8.3節で校正したものと同じ。
  # ⚠️ 既定の事前分布のままだと発散遷移が出ることを確認した。第8章と同じ
  # 事前分布・adapt_delta=0.95を試しても1件の発散遷移が残ったため、
  # adapt_delta=0.99・max_treedepth=12まで上げて0件になることを確認済み
  # （本文でモデルの当てはめ方を扱う場所ではないので、ここでは既に
  # 発散遷移ゼロを確認できた設定をそのまま使う）。
  tar_target(model_brm, brm(
    VOT ~ condition + (1 + condition | speaker) + (1 | word),
    data = vot_data,
    prior = c(
      prior(normal(50, 30), class = "Intercept"),
      prior(normal(0, 30), class = "b"),
      prior(exponential(0.1), class = "sd")
    ),
    chains = 4, iter = 2000, warmup = 1000, seed = 42,
    control = list(adapt_delta = 0.99, max_treedepth = 12)
  )),

  # 図表出力（モデルが変わったときだけ再描画される）
  tar_target(fig_vot_violin, {
    p <- ggplot(vot_data, aes(x = condition, y = VOT, fill = condition)) +
      geom_violin(alpha = 0.5) + theme_minimal(base_size = 14, base_family = "notosans")
    ggsave("fig_vot_violin.png", p, width = 6, height = 4)
    "fig_vot_violin.png"
  }, format = "file")
)
'
cat(pipeline_example, file = "_targets.R")
cat("_targets.R を書き出した。\n")


■ 実際に`tar_make()`を走らせる（1回目・コールドラン）：

初回はキャッシュが無いので、5つのターゲットすべてが計算される。brmsモデルのStanコンパイルとサンプリングが最も時間がかかる。

In [ ]:
t1 <- system.time(tar_make())
print(t1)

# --- 実行結果（実行例。所要時間は環境によって変わる） ---
# + vot_data_file dispatched
# ✔ vot_data_file completed
# + vot_data dispatched
# ✔ vot_data completed
# + model_brm dispatched
#   （Stanのコンパイルとサンプリング。4チェーン）
# ✔ model_brm completed [40.5s, 4.68 MB]
# + model_lmm dispatched
# ✔ model_lmm completed [26ms, 102.39 kB]
# + fig_vot_violin dispatched
# ✔ fig_vot_violin completed [185ms, 68.98 kB]
# ✔ ended pipeline [42.2s, 5 completed, 0 skipped]
#
# 【出力の読み方】
# ・5つのターゲットすべてに"dispatched"（実行された）と出ている。
#   コールドラン（キャッシュが何も無い状態）なので当然である。
# ・`model_brm`だけ突出して時間がかかっている（40.5秒。他は1秒未満）。
#   4チェーンのMCMCサンプリングが計算量の大半を占めるため、これが
#   「一度計算したら二度と計算し直したくない」ターゲットの典型である。
# ・**ここに出ている秒数はこの実行環境での1回の実行例であり、固定値では
#   ない。** CPUの速さやチェーンの並列化状況によって数倍変わりうる。
#   重要なのは絶対時間ではなく、次に見る2回目の実行との**相対的な差**である。


■ もう一度`tar_make()`を走らせる（2回目・変更なし）：

データもコードも何も変えずに再実行すると、全ターゲットが「最新（uptodate）」と判定され、何も計算し直さない。

In [ ]:
t2 <- system.time(tar_make())
print(t2)

# --- 実行結果（実行例） ---
# ✔ skipped pipeline [33ms, 5 skipped]
#
# 【出力の読み方】
# ・"skipped"が5件、"dispatched"は0件。**40.5秒かかったbrmsモデルの
#   再計算を含めて、パイプライン全体が33ミリ秒で終わっている。**
# ・これが`targets`の中心的な価値である。ノートブックを上から
#   再実行する方式だと、このために毎回40秒以上を払うことになる。
#   モデルが増え、1回の実行が数分・数十分になる規模の研究では、
#   この差が実務上決定的になる。


■ 入力を変えると何が起きるか（本文の問1に対応）：

`vot_f0_synthetic.csv`の中身を1箇所だけ書き換えて、再度`tar_make()`を走らせる。

In [ ]:
d <- read_csv("vot_f0_synthetic.csv", show_col_types = FALSE)
d$VOT[1] <- d$VOT[1] + 1   # ごく小さな変更
write_csv(d, "vot_f0_synthetic.csv")

t3 <- system.time(tar_make())
print(t3)

# --- 実行結果（実行例） ---
# + vot_data_file dispatched
# ✔ vot_data_file completed
# + vot_data dispatched
# ✔ vot_data completed
# + model_brm dispatched
# ✔ model_brm completed [48.9s, 4.68 MB]
# + model_lmm dispatched
# ✔ model_lmm completed
# + fig_vot_violin dispatched
# ✔ fig_vot_violin completed
# ✔ ended pipeline [50.9s, 5 completed, 0 skipped]
#
# 【出力の読み方】
# ・**今回は5つ全部が再計算されている。** 「ファイルの一部だけを変えたのに
#   なぜ全部再実行されるのか」と思うかもしれないが、これは正しい挙動である。
#   `model_lmm`も`model_brm`も`fig_vot_violin`も、依存を辿ればすべて
#   `vot_data_file`に行き着く（下のtar_manifest()参照）。**入力ファイルが
#   変わった以上、そこから下流のターゲットはすべて「古い」と判定するのが
#   `targets`の安全側の設計である。**
# ・「一部だけ再計算されるはず」という期待が外れるのは、たいてい
#   依存関係の作り方の勘違いである。図表出力だけを軽く直したい場合は、
#   モデルへの依存を切って別のターゲットに分けるなど、依存グラフの
#   設計そのものを見直す必要がある。


■ 依存関係を確認する（`tar_manifest()`・`tar_visnetwork()`）：

`tar_visnetwork()`はブラウザ上でクリック・ドラッグできる対話的なグラフを描く（Colab上でセルの出力として表示される）。本書は紙面上テキストで示すため、同じ依存関係を`tar_manifest()`で文字として確認する。

In [ ]:
print(tar_manifest(fields = c("name", "command")))

# 対話的なグラフはColab上でこのセルを実行すると表示される。
tar_visnetwork()

# --- 実行結果（tar_manifest()の部分） ---
# # A tibble: 5 x 2
#   name           command
#   <chr>          <chr>
# 1 vot_data_file  "vot_f0_synthetic.csv"
# 2 vot_data       "read_csv(vot_data_file, show_col_types = FALSE) %>% ..."
# 3 model_brm      "brm(VOT ~ condition + (1 + condition | speaker) + ..."
# 4 model_lmm      "lmer(VOT ~ condition + (1 + condition | speaker) + ..."
# 5 fig_vot_violin "{ p <- ggplot(vot_data, ...) ... }"
#
# 【出力の読み方】
# ・`command`列を見ると、`vot_data`・`model_brm`・`model_lmm`・
#   `fig_vot_violin`のすべてが、コードの中に`vot_data`（さらに遡れば
#   `vot_data_file`）への参照を持っている。**`targets`は関数の引数として
#   使われている変数名を静的に解析して依存グラフを作る**ため、
#   `_targets.R`の中で明示的に順番を書く必要がない。
# ・`tar_visnetwork()`はこの表と同じ情報をグラフとして描く。ノードの色で
#   「最新（緑）」「要再計算（青）」が一目でわかるので、大きな
#   パイプラインではこちらのほうが実用的である。


■ 何が嬉しいのか：

- `tar_make()`を実行すると、変更されたターゲットとその下流だけが再計算される。上で見たとおり、変更が無ければ40秒以上かかるbrmsモデルの計算が数十ミリ秒で終わる。
- `tar_visnetwork()`で依存関係をグラフとして可視化でき、「このモデルはどの前処理から来ているか」が一目でわかる。
- CI（継続的インテグレーション）に載せやすい。ノートブックのセルを手で順番に実行する必要がなくなる。

**ただし、これは学習段階で必要なものではない。** 本書の第0〜16章のように「1トピックを1回動かして理解する」場面では、単体のColabノートブックのほうがシンプルで学習効率が良い。パイプライン化は、同じ解析を何度も回す実際の研究フェーズに入ってから検討すればよい。

■ 演習：

1. 上のセルで実際に確認したとおり、`vot_f0_synthetic.csv`を書き換えると`model_lmm`・`model_brm`・`fig_vot_violin`がすべて再計算される。**`model_lmm`の`(1 + condition | speaker)`を`(1 | speaker)`に変えるだけ（データは変えない）場合、再計算されるのはどのターゲットだけになるか、予想してから`_targets.R`を書き換えて確認せよ。**
2. `tar_option_set(packages = ...)`に`ggplot2`を明示的に足す必要が無いのはなぜか（ヒント: `tidyverse`が読み込むパッケージ群を第1章で確認したことを思い出すこと）。

■ 演習の解答（問1）：

`_targets.R`の`model_lmm`ターゲットの式だけを`(1 + condition | speaker)`から`(1 | speaker)`に書き換えて`tar_make()`を実行すると、実際に**`model_lmm`だけ**が再計算される（`✔ ended pipeline [1 completed, 4 skipped]`）。`vot_data_file`・`vot_data`は入力もコードも変えていないのでuptodateのまま、`model_brm`もコードを変えていないので再計算されない。`fig_vot_violin`は`vot_data`から作図しているだけで`model_lmm`には依存していないので、これも再計算されない。**依存グラフに実際に含まれる関係だけが再計算の範囲を決める**——同じスクリプトの中に書かれているというだけでは、依存関係にはならない。